# CardioSentinel — GPU training on Colab

Runs the real (non-smoke) training configs for both experiments. Do this first:

**Runtime → Change runtime type → T4 GPU** (or better), then run cells top to bottom.

Everything here was proven correct on CPU first via `scripts/smoke_test.py` and a small-scale
real-data run — this notebook is only about getting the *real* configs onto a GPU, not
re-deriving the pipeline.


In [ ]:
!nvidia-smi

## 1. Get the code onto this VM

Pick ONE of these:

- **(A) GitHub** — if you've pushed `CardioSenseFinal` to a repo, uncomment and edit the clone line.
- **(B) Google Drive** — zip the `cardiosentinel/` folder locally, upload it into a Drive folder,
  mount Drive below, and unzip from there. Good if you don't want a public/private repo yet.
- **(C) Direct upload** — use the Colab file browser (folder icon, left sidebar) to drag the
  `cardiosentinel/` folder in directly. Fine for a one-off run, but you'll re-upload every session.

This notebook assumes the code ends up at `/content/cardiosentinel`.


In [ ]:
# --- Option A: GitHub ---
# !git clone https://github.com/<your-username>/<your-repo>.git /content/repo
# %cd /content/repo/cardiosentinel

# --- Option B: Google Drive ---
# from google.colab import drive
# drive.mount('/content/drive')
# !unzip -q "/content/drive/MyDrive/cardiosentinel.zip" -d /content
# %cd /content/cardiosentinel

# --- Option C: direct upload ---
# Drag the cardiosentinel/ folder into the Colab file browser under /content/,
# then just:
# %cd /content/cardiosentinel

import os
assert os.path.isdir("src"), "Not inside cardiosentinel/ — pick one of the options above first."
print("cwd OK:", os.getcwd())

## 2. Install dependencies

In [ ]:
!pip install -q -r requirements.txt

## 3. Kaggle credentials

Paste your Kaggle credential when prompted (input is hidden). Use whichever you have:

- A **classic API token** (`kaggle.json` — has `username` + `key`): paste the **key** value when
  prompted below, and also set your username in the cell.
- A **newer per-account API token** (starts with `KGAT_`): this is what the download script in
  this repo (`scripts/kaggle_download.py`) expects — it authenticates directly against Kaggle's
  REST API with this token, because the `kaggle` PyPI package (as of 1.7.4.5) doesn't support
  this newer token type yet, only the classic username/key pair.


In [ ]:
import getpass, os
from pathlib import Path

token = getpass.getpass("Paste your Kaggle API token (KGAT_... style): ").strip()
kaggle_dir = Path.home() / ".kaggle"
kaggle_dir.mkdir(exist_ok=True)
(kaggle_dir / "access_token").write_text(token)
os.chmod(kaggle_dir / "access_token", 0o600)
print("Saved to", kaggle_dir / "access_token")

# If you have a CLASSIC kaggle.json instead (username + key), use this instead:
# import json
# username = input("Kaggle username: ")
# key = getpass.getpass("Kaggle API key: ")
# (kaggle_dir / "kaggle.json").write_text(json.dumps({"username": username, "key": key}))
# os.chmod(kaggle_dir / "kaggle.json", 0o600)

## 4. Download the datasets

Same script used locally — downloads all 5 dataset folders into `../data/` relative to
`cardiosentinel/`. Colab's bandwidth is much better than a typical home connection, so this
should be a lot faster than it was locally.


In [ ]:
!python scripts/kaggle_download.py

In [ ]:
!python scripts/inspect_schemas.py

## 5. Sanity check before committing GPU time

Fast CPU-only architecture check — confirms nothing broke in transit (imports, torch version,
etc.) before spending GPU time on a real run.


In [ ]:
!python scripts/smoke_test.py

## 6. Real training — Experiment 1: trimodal deterioration model

Fine-tunes ClinicalBERT + BioBERT for real (`freeze_text_encoders: false` in
`configs/trimodal.yaml`) — this is the run that wasn't practical on CPU.

If DataLoader workers hang or crash (rare, depends on Colab's CPU allocation), lower
`num_workers` in `configs/trimodal.yaml` to 2.


In [ ]:
!python -m src.train_trimodal --config configs/trimodal.yaml

### Ablations (optional, run after the baseline above looks right)

In [ ]:
!bash scripts/run_ablations.sh configs/trimodal.yaml

## 7. Real training — Experiment 2: ECG risk classifier

Separate model, separate data — PTB-XL signal + PTB-XL+ features/demographics.


In [ ]:
!python -m src.train_ecg --config configs/ecg.yaml

## 8. Persist checkpoints before the session ends

Colab VMs are ephemeral — anything not saved to Drive disappears when the runtime recycles.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/cardiosentinel_checkpoints
!cp -r checkpoints/* /content/drive/MyDrive/cardiosentinel_checkpoints/
print("Checkpoints copied to Google Drive.")